##Climate Trace Dataset Investigation

Source: https://huggingface.co/datasets/tjhunter/climate-trace/tree/main/v3-2024-ct5

In [2]:
!pip install pyspark

In [6]:
# Dependencies

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, when
from pyspark.sql.types import StringType
from pyspark.sql.functions import countDistinct
from IPython.display import display

# Create a SparkSession
spark = SparkSession.builder.appName("ClimateTrace").getOrCreate()

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# Parquet Files
parquet_files = {
    "2021_ch4.parquet" : 2021,
    "2021_co2.parquet" : 2021,
    "2021_n2o.parquet" : 2021,
    "2021_co2e_100yr.parquet" : 2021,
    "2022_ch4.parquet" : 2022,
    "2022_co2.parquet" : 2022,
    "2022_n2o.parquet" : 2022,
    "2022_co2e_100yr.parquet" : 2022,
    "2023_ch4.parquet" : 2023,
    "2023_co2.parquet" : 2023,
    "2023_n2o.parquet" : 2023,
    "2023_co2e_100yr.parquet" : 2023,
    "2024_ch4.parquet" : 2024,
    "2024_co2.parquet" : 2024,
    "2024_n2o.parquet" : 2024,
    "2024_co2e_100yr.parquet" : 2024
}
print("Parquet Files in the directory:\n")
for each in parquet_files:
    print(each)

target_parquet = input("Enter the target parquet file name to upload: ")
df = None

if target_parquet in parquet_files:
    parquet_file = f"/content/drive/MyDrive/School Projects/Climate Trace Analysis/{target_parquet}"
    df = spark.read.parquet(parquet_file)
    print("Parquet file loaded successfully.")
else:
    print("Invalid input. Please enter a valid parquet file name.")

Parquet Files in the directory:

2021_ch4.parquet
2021_co2.parquet
2021_n2o.parquet
2021_co2e_100yr.parquet
2022_ch4.parquet
2022_co2.parquet
2022_n2o.parquet
2022_co2e_100yr.parquet
2023_ch4.parquet
2023_co2.parquet
2023_n2o.parquet
2023_co2e_100yr.parquet
2024_ch4.parquet
2024_co2.parquet
2024_n2o.parquet
2024_co2e_100yr.parquet
Enter the target parquet file name to upload: 2021_ch4.parquet
Parquet file loaded successfully.


In [8]:
# Schema Exploration
schema = df.schema

for field in schema:
    print(field)

StructField('source_id', DecimalType(20,0), True)
StructField('iso3_country', StringType(), True)
StructField('sector', StringType(), True)
StructField('subsector', StringType(), True)
StructField('original_inventory_sector', StringType(), True)
StructField('start_time', TimestampType(), True)
StructField('end_time', TimestampType(), True)
StructField('temporal_granularity', StringType(), True)
StructField('gas', StringType(), True)
StructField('emissions_quantity', DoubleType(), True)
StructField('emissions_factor', DoubleType(), True)
StructField('emissions_factor_units', StringType(), True)
StructField('capacity', DoubleType(), True)
StructField('capacity_units', StringType(), True)
StructField('capacity_factor', DoubleType(), True)
StructField('activity', DoubleType(), True)
StructField('activity_units', StringType(), True)
StructField('created_date', TimestampType(), True)
StructField('modified_date', TimestampType(), True)
StructField('source_name', StringType(), True)
StructFiel

## <span style="color: white; font-weight: bold; text-decoration: underline;">source_id</span>
The source id is the unique identifier for the source of the data.

Problems:
1. uses uint64 that can be converted to more efficient int64

Data Type: `decimal`

**Status: <span style="color: red;"> READY
</span>**

In [67]:
# Source ID Exploration

column_title = "Source ID"
column_name = "source_id"

# Data Type
print(f"{column_title} Data Type")
data_type = df.schema[column_name].dataType
print(data_type)

# Head
print(f"\n{column_title} Head Values:")
df.select(column_name).show(5)

# Tail
print(f"{column_title} Tail Values:")
df.select(column_name).orderBy(column_name, ascending=False).show(5)

# Min and Max
if not isinstance(data_type, StringType):
    print(f"{column_title} Max Value:")
    df.agg({column_name : "max"}).show()

    print(f"{column_title} Min Value:")
    df.agg({column_name : "min"}).show()

# Distinct
if isinstance(data_type, StringType):
    print(f"{column_title} Distinct Values:")
    df.select(column_name).distinct().show()

# Total Count
print(f"{column_title} Count:")
print(df.select(column_name).count())

# Value Count
print(f"\n{column_title} Value Counts:")
df.groupBy(column_name).count().show()

# Null Count
print(f"{column_title} Null Count:")
print(df.filter(col(column_name).isNull()).count())

Source ID Data Type
DecimalType(20,0)
Source ID Head Values:
+---------+
|source_id|
+---------+
| 11082620|
| 11082620|
| 11082620|
| 11082620|
| 11082721|
+---------+
only showing top 5 rows

Source ID Tail Values:
+---------+
|source_id|
+---------+
| 38339430|
| 38339430|
| 38339430|
| 38339430|
| 38339430|
+---------+
only showing top 5 rows

Source ID Max Value:
+--------------+
|max(source_id)|
+--------------+
|      38339430|
+--------------+

Source ID Min Value:
+--------------+
|min(source_id)|
+--------------+
|           110|
+--------------+

Source ID Count:
15184500

Source ID Value Counts:
+---------+-----+
|source_id|count|
+---------+-----+
| 11097298|   12|
| 37215827|   12|
| 11127731|   12|
| 11131222|   12|
| 11115712|   12|
| 11128080|   12|
| 10721281|   12|
| 10778725|   12|
| 10843369|   12|
| 10824069|   12|
| 10859706|   12|
| 11248289|   12|
| 11278174|   12|
| 11293635|   12|
| 11293472|   12|
| 11295944|   12|
| 37177796|   12|
| 10932135|   12|
| 10920

## <span style="color: white; font-weight: bold; text-decoration: underline;">iso3_country</span>
The ISO 3 country code is a three-letter code that is used to identify countries.

Data Type: `string`

**Status: <span style="color: red;">NOT READY</span>**

Problems:
1. there is an invalid country code in some dataset parquet

In [68]:

column_title = "iso3 Country Code"
column_name = "iso3_country"

# Data Type
print(f"{column_title} Data Type")
data_type = df.schema[column_name].dataType
print(data_type)

# Head
print(f"\n{column_title} Head Values:")
df.select(column_name).show(5)

# Tail
print(f"{column_title} Tail Values:")
df.select(column_name).orderBy(column_name, ascending=False).show(5)

# Min and Max
if not isinstance(data_type, StringType):
    print(f"{column_title} Max Value:")
    df.agg({column_name : "max"}).show()

    print(f"{column_title} Min Value:")
    df.agg({column_name : "min"}).show()

# Distinct
if isinstance(data_type, StringType):
    print(f"{column_title} Distinct Values:")
    df.select(column_name).distinct().show()

# Total Count
print(f"{column_title} Count:")
print(df.select(column_name).count())

# Value Count
print(f"\n{column_title} Value Counts:")
df.groupBy(column_name).count().show()

# Null Count
print(f"{column_title} Null Count:")
print(df.filter(col(column_name).isNull()).count())

iso3 Country Code Data Type
StringType()

iso3 Country Code Head Values:
+------------+
|iso3_country|
+------------+
|         KGZ|
|         KGZ|
|         KGZ|
|         KGZ|
|         KGZ|
+------------+
only showing top 5 rows

iso3 Country Code Tail Values:
+------------+
|iso3_country|
+------------+
|         ZWE|
|         ZWE|
|         ZWE|
|         ZWE|
|         ZWE|
+------------+
only showing top 5 rows

iso3 Country Code Distinct Values:
+------------+
|iso3_country|
+------------+
|         NIU|
|         CCK|
|         HTI|
|         PSE|
|         LVA|
|         BRB|
|         POL|
|         ZMB|
|         JAM|
|         BRA|
|         SPM|
|         ARM|
|         MOZ|
|         JOR|
|         CUB|
|         FRA|
|         SOM|
|         ABW|
|         TCA|
|         COD|
+------------+
only showing top 20 rows

iso3 Country Code Count:
15184500

iso3 Country Code Value Counts:
+------------+-------+
|iso3_country|  count|
+------------+-------+
|         NIU|    1

## <span style="color: white; font-weight: bold; text-decoration: underline;">sector</span>
The sector is the economic sector that the activity belongs to

Data Type: `string`

**Status: <span style="color: red;">READY</span>**

Problems: None

In [69]:
# Sector Exploration
column_title = "Sector"
column_name = "sector"


# Data Type
print(f"{column_title} Data Type")
data_type = df.schema[column_name].dataType
print(data_type)

# Head
print(f"\n{column_title} Head Values:")
df.select(column_name).show(5)

# Tail
print(f"{column_title} Tail Values:")
df.select(column_name).orderBy(column_name, ascending=False).show(5)

# Min and Max
if not isinstance(data_type, StringType):
    print(f"{column_title} Max Value:")
    df.agg({column_name : "max"}).show()

    print(f"{column_title} Min Value:")
    df.agg({column_name : "min"}).show()

# Distinct
if isinstance(data_type, StringType):
    print(f"{column_title} Distinct Values:")
    df.select(column_name).distinct().show()

# Total Count
print(f"{column_title} Count:")
print(df.select(column_name).count())

# Value Count
print(f"\n{column_title} Value Counts:")
df.groupBy(column_name).count().show()

# Null Count
print(f"{column_title} Null Count:")
print(df.filter(col(column_name).isNull()).count())

Sector Data Type
StringType()

Sector Head Values:
+-----------+
|     sector|
+-----------+
|agriculture|
|agriculture|
|agriculture|
|agriculture|
|agriculture|
+-----------+
only showing top 5 rows

Sector Tail Values:
+------+
|sector|
+------+
| waste|
| waste|
| waste|
| waste|
| waste|
+------+
only showing top 5 rows

Sector Distinct Values:
+--------------------+
|              sector|
+--------------------+
|  mineral-extraction|
|               power|
|         agriculture|
|fossil-fuel-opera...|
|               waste|
|       manufacturing|
|forestry-and-land...|
|      transportation|
|           buildings|
+--------------------+

Sector Count:
15184500

Sector Value Counts:
+--------------------+-------+
|              sector|  count|
+--------------------+-------+
|  mineral-extraction|  19908|
|               power| 106440|
|         agriculture|5213832|
|fossil-fuel-opera...|  60804|
|               waste| 777072|
|       manufacturing| 423876|
|forestry-and-land...|61

### <span style="color: white; font-weight: bold; text-decoration: underline;">subsector</span>
The subsector is the economic subsector that the activity belongs to.

Data Type: `string`

**Status: <span style="color: green;">READY</span>**

Problems: None

In [70]:
# Subsector Exploration
column_title = "Subsector"
column_name = "subsector"

# Data Type
print(f"{column_title} Data Type")
data_type = df.schema[column_name].dataType
print(data_type)

# Head
print(f"\n{column_title} Head Values:")
df.select(column_name).show(5)

# Tail
print(f"{column_title} Tail Values:")
df.select(column_name).orderBy(column_name, ascending=False).show(5)

# Min and Max
if not isinstance(data_type, StringType):
    print(f"{column_title} Max Value:")
    df.agg({column_name : "max"}).show()

    print(f"{column_title} Min Value:")
    df.agg({column_name : "min"}).show()

# Distinct
if isinstance(data_type, StringType):
    print(f"{column_title} Distinct Values:")
    df.select(column_name).distinct().show()

# Total Count
print(f"{column_title} Count:")
print(df.select(column_name).count())

# Value Count
print(f"\n{column_title} Value Counts:")
df.groupBy(column_name).count().show()

# Null Count
print(f"{column_title} Null Count:")
print(df.filter(col(column_name).isNull()).count())

Subsector Data Type
StringType()

Subsector Head Values:
+--------------+
|     subsector|
+--------------+
|cropland-fires|
|cropland-fires|
|cropland-fires|
|cropland-fires|
|cropland-fires|
+--------------+
only showing top 5 rows

Subsector Tail Values:
+-------------+
|    subsector|
+-------------+
|wetland-fires|
|wetland-fires|
|wetland-fires|
|wetland-fires|
|wetland-fires|
+-------------+
only showing top 5 rows

Subsector Distinct Values:
+--------------------+
|           subsector|
+--------------------+
|      cropland-fires|
|         net-wetland|
|      bauxite-mining|
|            aluminum|
|   domestic-aviation|
|food-beverage-tob...|
|      iron-and-steel|
|               glass|
|           chemicals|
|industrial-wastew...|
|      net-shrubgrass|
|                lime|
|         iron-mining|
|forest-land-degra...|
|international-avi...|
|enteric-fermentat...|
|     net-forest-land|
|domestic-wastewat...|
|enteric-fermentat...|
|         coal-mining|
+----------------

### <span style="color: white; font-weight: bold; text-decoration: underline;">original_inventory_sector</span>
The original inventory sector is the economic sector that the activity belongs to.

Data Type: `string`

**Status: <span style="color: red;">NOT READY</span>**

Problems:
1. does not have any value across all the datasets

In [71]:
# Original Inventory Sector Exploration
column_title = "Original Inventory"
column_name = "original_inventory_sector"

# Data Type
print(f"{column_title} Data Type")
data_type = df.schema[column_name].dataType
print(data_type)

# Head
print(f"\n{column_title} Head Values:")
df.select(column_name).show(5)

# Tail
print(f"{column_title} Tail Values:")
df.select(column_name).orderBy(column_name, ascending=False).show(5)

# Min and Max
if not isinstance(data_type, StringType):
    print(f"{column_title} Max Value:")
    df.agg({column_name : "max"}).show()

    print(f"{column_title} Min Value:")
    df.agg({column_name : "min"}).show()

# Distinct
if isinstance(data_type, StringType):
    print(f"{column_title} Distinct Values:")
    df.select(column_name).distinct().show()

# Total Count
print(f"{column_title} Count:")
print(df.select(column_name).count())

# Value Count
print(f"\n{column_title} Value Counts:")
df.groupBy(column_name).count().show()

# Null Count
print(f"{column_title} Null Count:")
print(df.filter(col(column_name).isNull()).count())

Original Inventory Data Type
StringType()

Original Inventory Head Values:
+-------------------------+
|original_inventory_sector|
+-------------------------+
|                     NULL|
|                     NULL|
|                     NULL|
|                     NULL|
|                     NULL|
+-------------------------+
only showing top 5 rows

Original Inventory Tail Values:
+-------------------------+
|original_inventory_sector|
+-------------------------+
|                     NULL|
|                     NULL|
|                     NULL|
|                     NULL|
|                     NULL|
+-------------------------+
only showing top 5 rows

Original Inventory Distinct Values:
+-------------------------+
|original_inventory_sector|
+-------------------------+
|                     NULL|
+-------------------------+

Original Inventory Count:
15184500

Original Inventory Value Counts:
+-------------------------+--------+
|original_inventory_sector|   count|
+------------------

### <span style="color: white; font-weight: bold; text-decoration: underline;">start_time</span>
The start time of the activity.

Data Type: `timestamp`

**Status: <span style="color: green;">READY</span>**

Problems: None

In [72]:
# Start Time Exploration
column_title = "Start Time"
column_name = "start_time"

# Data Type
print(f"{column_title} Data Type")
data_type = df.schema[column_name].dataType
print(data_type)

# Head
print(f"\n{column_title} Head Values:")
df.select(column_name).show(5)

# Tail
print(f"{column_title} Tail Values:")
df.select(column_name).orderBy(column_name, ascending=False).show(5)

# Min and Max
if not isinstance(data_type, StringType):
    print(f"{column_title} Max Value:")
    df.agg({column_name : "max"}).show()

    print(f"{column_title} Min Value:")
    df.agg({column_name : "min"}).show()

# Distinct
if isinstance(data_type, StringType):
    print(f"{column_title} Distinct Values:")
    df.select(column_name).distinct().show()

# Total Count
print(f"{column_title} Count:")
print(df.select(column_name).count())

# Value Count
print(f"\n{column_title} Value Counts:")
df.groupBy(column_name).count().show()

# Null Count
print(f"{column_title} Null Count:")
print(df.filter(col(column_name).isNull()).count())

Start Time Data Type
TimestampType()

Start Time Head Values:
+-------------------+
|         start_time|
+-------------------+
|2021-09-01 00:00:00|
|2021-10-01 00:00:00|
|2021-11-01 00:00:00|
|2021-12-01 00:00:00|
|2021-01-01 00:00:00|
+-------------------+
only showing top 5 rows

Start Time Tail Values:
+-------------------+
|         start_time|
+-------------------+
|2021-12-01 00:00:00|
|2021-12-01 00:00:00|
|2021-12-01 00:00:00|
|2021-12-01 00:00:00|
|2021-12-01 00:00:00|
+-------------------+
only showing top 5 rows

Start Time Max Value:
+-------------------+
|    max(start_time)|
+-------------------+
|2021-12-01 00:00:00|
+-------------------+

Start Time Min Value:
+-------------------+
|    min(start_time)|
+-------------------+
|2021-01-01 00:00:00|
+-------------------+

Start Time Count:
15184500

Start Time Value Counts:
+-------------------+-------+
|         start_time|  count|
+-------------------+-------+
|2021-07-01 00:00:00|1265375|
|2021-03-01 00:00:00|1265375|

### <span style="color: white; font-weight: bold; text-decoration: underline;">end_time</span>
The end time of the activity.

Data Type: `timestamp`

**Status: <span style="color: green;">READY</span>**

Problems: None

In [73]:
# End Time Exploration
column_title = "End Time"
column_name = "end_time"

# Data Type
print(f"{column_title} Data Type")
data_type = df.schema[column_name].dataType
print(data_type)

# Head
print(f"\n{column_title} Head Values:")
df.select(column_name).show(5)

# Tail
print(f"{column_title} Tail Values:")
df.select(column_name).orderBy(column_name, ascending=False).show(5)

# Min and Max
if not isinstance(data_type, StringType):
    print(f"{column_title} Max Value:")
    df.agg({column_name : "max"}).show()

    print(f"{column_title} Min Value:")
    df.agg({column_name : "min"}).show()

# Distinct
if isinstance(data_type, StringType):
    print(f"{column_title} Distinct Values:")
    df.select(column_name).distinct().show()

# Total Count
print(f"{column_title} Count:")
print(df.select(column_name).count())

# Value Count
print(f"\n{column_title} Value Counts:")
df.groupBy(column_name).count().show()

# Null Count
print(f"{column_title} Null Count:")
print(df.filter(col(column_name).isNull()).count())

End Time Data Type
TimestampType()

End Time Head Values:
+-------------------+
|           end_time|
+-------------------+
|2021-09-30 00:00:00|
|2021-10-31 00:00:00|
|2021-11-30 00:00:00|
|2021-12-31 00:00:00|
|2021-01-31 00:00:00|
+-------------------+
only showing top 5 rows

End Time Tail Values:
+-------------------+
|           end_time|
+-------------------+
|2021-12-31 00:00:00|
|2021-12-31 00:00:00|
|2021-12-31 00:00:00|
|2021-12-31 00:00:00|
|2021-12-31 00:00:00|
+-------------------+
only showing top 5 rows

End Time Max Value:
+-------------------+
|      max(end_time)|
+-------------------+
|2021-12-31 00:00:00|
+-------------------+

End Time Min Value:
+-------------------+
|      min(end_time)|
+-------------------+
|2021-01-31 00:00:00|
+-------------------+

End Time Count:
15184500

End Time Value Counts:
+-------------------+-------+
|           end_time|  count|
+-------------------+-------+
|2021-09-30 00:00:00|1265375|
|2021-01-31 00:00:00|1265375|
|2021-03-31 0

### <span style="color: white; font-weight: bold; text-decoration: underline;">temporal_granularity</span>
The temporal granularity of the activity.

Data Type: `string`

**Status: <span style="color: green;">READY</span>**

Problems: None

In [74]:
# Temporal Granularity Exploration
column_title = "Temporal Granularity"
column_name = "temporal_granularity"

# Data Type
print(f"{column_title} Data Type")
data_type = df.schema[column_name].dataType
print(data_type)

# Head
print(f"\n{column_title} Head Values:")
df.select(column_name).show(5)

# Tail
print(f"{column_title} Tail Values:")
df.select(column_name).orderBy(column_name, ascending=False).show(5)

# Min and Max
if not isinstance(data_type, StringType):
    print(f"{column_title} Max Value:")
    df.agg({column_name : "max"}).show()

    print(f"{column_title} Min Value:")
    df.agg({column_name : "min"}).show()

# Distinct
if isinstance(data_type, StringType):
    print(f"{column_title} Distinct Values:")
    df.select(column_name).distinct().show()

# Total Count
print(f"{column_title} Count:")
print(df.select(column_name).count())

# Value Count
print(f"\n{column_title} Value Counts:")
df.groupBy(column_name).count().show()

# Null Count
print(f"{column_title} Null Count:")
print(df.filter(col(column_name).isNull()).count())

Temporal Granularity Data Type
StringType()

Temporal Granularity Head Values:
+--------------------+
|temporal_granularity|
+--------------------+
|               month|
|               month|
|               month|
|               month|
|               month|
+--------------------+
only showing top 5 rows

Temporal Granularity Tail Values:
+--------------------+
|temporal_granularity|
+--------------------+
|               month|
|               month|
|               month|
|               month|
|               month|
+--------------------+
only showing top 5 rows

Temporal Granularity Distinct Values:
+--------------------+
|temporal_granularity|
+--------------------+
|               month|
+--------------------+

Temporal Granularity Count:
15184500

Temporal Granularity Value Counts:
+--------------------+--------+
|temporal_granularity|   count|
+--------------------+--------+
|               month|15184500|
+--------------------+--------+

Temporal Granularity Null Count:
0


### <span style="color: white; font-weight: bold; text-decoration: underline;">gas</span>
The gas type observed in the activity.

Data Type: `string`

**Status: <span style="color: green;">READY</span>**

Problems:
1. label should be renamed to something more descriptive like gas type

In [75]:
# Gas Type Exploration
column_title = "Gas Type"
column_name = "gas"

# Data Type
print(f"{column_title} Data Type")
data_type = df.schema[column_name].dataType
print(data_type)

# Head
print(f"\n{column_title} Head Values:")
df.select(column_name).show(5)

# Tail
print(f"{column_title} Tail Values:")
df.select(column_name).orderBy(column_name, ascending=False).show(5)

# Min and Max
if not isinstance(data_type, StringType):
    print(f"{column_title} Max Value:")
    df.agg({column_name : "max"}).show()

    print(f"{column_title} Min Value:")
    df.agg({column_name : "min"}).show()

# Distinct
if isinstance(data_type, StringType):
    print(f"{column_title} Distinct Values:")
    df.select(column_name).distinct().show()

# Total Count
print(f"{column_title} Count:")
print(df.select(column_name).count())

# Value Count
print(f"\n{column_title} Value Counts:")
df.groupBy(column_name).count().show()

# Null Count
print(f"{column_title} Null Count:")
print(df.filter(col(column_name).isNull()).count())

Gas Type Data Type
StringType()

Gas Type Head Values:
+---+
|gas|
+---+
|ch4|
|ch4|
|ch4|
|ch4|
|ch4|
+---+
only showing top 5 rows

Gas Type Tail Values:
+---+
|gas|
+---+
|ch4|
|ch4|
|ch4|
|ch4|
|ch4|
+---+
only showing top 5 rows

Gas Type Distinct Values:
+---+
|gas|
+---+
|ch4|
+---+

Gas Type Count:
15184500

Gas Type Value Counts:
+---+--------+
|gas|   count|
+---+--------+
|ch4|15184500|
+---+--------+

Gas Type Null Count:
0


### <span style="color: white; font-weight: bold; text-decoration: underline;">emissions_quantity</span>
The emissions quantity of the activity.

Data Type: `double`

**Status: <span style="color: green;">READY</span>**

Problems: None

In [76]:
# Emissions Quantity Exploration
column_title = "Emissions Quantity"
column_name = "emissions_quantity"

# Data Type
print(f"{column_title} Data Type")
data_type = df.schema[column_name].dataType
print(data_type)

# Head
print(f"\n{column_title} Head Values:")
df.select(column_name).show(5)

# Tail
print(f"{column_title} Tail Values:")
df.select(column_name).orderBy(column_name, ascending=False).show(5)

# Min and Max
if not isinstance(data_type, StringType):
    print(f"{column_title} Max Value:")
    df.agg({column_name : "max"}).show()

    print(f"{column_title} Min Value:")
    df.agg({column_name : "min"}).show()

# Distinct
if isinstance(data_type, StringType):
    print(f"{column_title} Distinct Values:")
    df.select(column_name).distinct().show()

# Total Count
print(f"{column_title} Count:")
print(df.select(column_name).count())

# Value Count
print(f"\n{column_title} Value Counts:")
df.groupBy(column_name).count().show()

# Null Count
print(f"{column_title} Null Count:")
print(df.filter(col(column_name).isNull()).count())

Emissions Quantity Data Type
DoubleType()

Emissions Quantity Head Values:
+------------------+
|emissions_quantity|
+------------------+
|1.7397360595540778|
| 2.421369167131417|
| 1.637782823959021|
|0.5098989112747037|
|0.1230362365371346|
+------------------+
only showing top 5 rows

Emissions Quantity Tail Values:
+------------------+
|emissions_quantity|
+------------------+
|1006126.2679489036|
| 991152.2025790858|
|  959186.759272008|
| 786601.1066666663|
| 786601.1066666663|
+------------------+
only showing top 5 rows

Emissions Quantity Max Value:
+-----------------------+
|max(emissions_quantity)|
+-----------------------+
|     1006126.2679489036|
+-----------------------+

Emissions Quantity Min Value:
+-----------------------+
|min(emissions_quantity)|
+-----------------------+
|                    0.0|
+-----------------------+

Emissions Quantity Count:
15184500

Emissions Quantity Value Counts:
+------------------+-----+
|emissions_quantity|count|
+------------------+

### <span style="color: white; font-weight: bold; text-decoration: underline;">emissions_factor</span>
The emissions factor of the activity.

Data Type: `double`

**Status: <span style="color: green;">NOT READY</span>**

Problems:
1. Has null or missing values

In [77]:
# Emissions Factor Exploration
column_title = "Emissions Factor"
column_name = "emissions_factor"

# Data Type
print(f"{column_title} Data Type")
data_type = df.schema[column_name].dataType
print(data_type)

# Head
print(f"\n{column_title} Head Values:")
df.select(column_name).show(5)

# Tail
print(f"{column_title} Tail Values:")
df.select(column_name).orderBy(column_name, ascending=False).show(5)

# Min and Max
if not isinstance(data_type, StringType):
    print(f"{column_title} Max Value:")
    df.agg({column_name : "max"}).show()

    print(f"{column_title} Min Value:")
    df.agg({column_name : "min"}).show()

# Distinct
if isinstance(data_type, StringType):
    print(f"{column_title} Distinct Values:")
    df.select(column_name).distinct().show()

# Total Count
print(f"{column_title} Count:")
print(df.select(column_name).count())

# Value Count
print(f"\n{column_title} Value Counts:")
df.groupBy(column_name).count().show()

# Null Count
print(f"{column_title} Null Count:")
print(df.filter(col(column_name).isNull()).count())

Emissions Factor Data Type
DoubleType()

Emissions Factor Head Values:
+----------------+
|emissions_factor|
+----------------+
|          0.0023|
|          0.0023|
|          0.0023|
|          0.0023|
|          0.0023|
+----------------+
only showing top 5 rows

Emissions Factor Tail Values:
+----------------+
|emissions_factor|
+----------------+
|         5261.26|
|         5261.26|
|         5261.26|
|         5261.26|
|         5261.26|
+----------------+
only showing top 5 rows

Emissions Factor Max Value:
+---------------------+
|max(emissions_factor)|
+---------------------+
|              5261.26|
+---------------------+

Emissions Factor Min Value:
+---------------------+
|min(emissions_factor)|
+---------------------+
|                  0.0|
+---------------------+

Emissions Factor Count:
15184500

Emissions Factor Value Counts:
+--------------------+-----+
|    emissions_factor|count|
+--------------------+-----+
|   1.953000013768E-4|    2|
|   1.952999999759E-4|   10|

### <span style="color: white; font-weight: bold; text-decoration: underline;">capacity</span>
The gas type observed in the activity.

Data Type: `double`

**Status: <span style="color: red;">NOT READY</span>**

Problems:
1. has null or missing values
2. has infinity value

In [53]:
# Capacity Exploration
column_title = "Capacity"
column_name = "capacity"

# Data Type
print(f"{column_title} Data Type")
data_type = df.schema[column_name].dataType
print(data_type)

# Head
print(f"\n{column_title} Head Values:")
df.select(column_name).show(5)

# Tail
print(f"{column_title} Tail Values:")
df.select(column_name).orderBy(column_name, ascending=False).show(5)

# Min and Max
if not isinstance(data_type, StringType):
    print(f"{column_title} Max Value:")
    df.agg({column_name : "max"}).show()

    print(f"{column_title} Min Value:")
    df.agg({column_name : "min"}).show()

# Distinct
if isinstance(data_type, StringType):
    print(f"{column_title} Distinct Values:")
    df.select(column_name).distinct().show()

# Total Count
print(f"{column_title} Count:")
print(df.select(column_name).count())

# Value Count
print(f"\n{column_title} Value Counts:")
df.groupBy(column_name).count().show()

# Null Count
print(f"{column_title} Null Count:")
print(df.filter(col(column_name).isNull()).count())

Capacity Data Type
DoubleType()
Capacity Head Values:
+-----------------+
|         capacity|
+-----------------+
| 5301.96148042613|
| 5301.96148042613|
| 5301.96148042613|
| 5301.96148042613|
|6030.153967362565|
+-----------------+
only showing top 5 rows

Capacity Tail Values:
+--------+
|capacity|
+--------+
|Infinity|
|Infinity|
|Infinity|
|Infinity|
|Infinity|
+--------+
only showing top 5 rows

Capacity Count:
15184500
Capacity Value Counts:
+------------------+-----+
|          capacity|count|
+------------------+-----+
|4217.6684399571695|   12|
|100945.90280949564|   12|
| 58.00026352808783|   12|
|  585.257357291574|   12|
|  50383.3640869687|   12|
| 87480.62672733956|   12|
|124949.84508174176|   12|
| 819280.8600828913|   12|
|1600.2870372706695|   12|
| 26.80405370168061|   12|
| 522.5614839493696|   12|
|19.225170125755827|   12|
|1413.5574512874305|   12|
|247.64055251986343|   12|
| 4849.159447174913|   12|
| 3783.095075718552|   12|
| 952.7167832937564|   12|
| 40534

### <span style="color: white; font-weight: bold; text-decoration: underline;">activity</span>
The gas type observed in the activity.

Data Type: `double`

**Status: <span style="color: green;">READY</span>**

Problems:
1. has null or missing values

In [54]:
# Activity Exploration
column_title = "Activity"
column_name = "activity"

# Data Type
print(f"{column_title} Data Type")
data_type = df.schema[column_name].dataType
print(data_type)

# Head
print(f"\n{column_title} Head Values:")
df.select(column_name).show(5)

# Tail
print(f"{column_title} Tail Values:")
df.select(column_name).orderBy(column_name, ascending=False).show(5)

# Min and Max
if not isinstance(data_type, StringType):
    print(f"{column_title} Max Value:")
    df.agg({column_name : "max"}).show()

    print(f"{column_title} Min Value:")
    df.agg({column_name : "min"}).show()

# Distinct
if isinstance(data_type, StringType):
    print(f"{column_title} Distinct Values:")
    df.select(column_name).distinct().show()

# Total Count
print(f"{column_title} Count:")
print(df.select(column_name).count())

# Value Count
print(f"\n{column_title} Value Counts:")
df.groupBy(column_name).count().show()

# Null Count
print(f"{column_title} Null Count:")
print(df.filter(col(column_name).isNull()).count())

Activity Data Type
DoubleType()
Activity Head Values:
+------------------+
|          activity|
+------------------+
| 756.4069824148164|
| 1052.769203100616|
| 712.0794886778352|
|221.69517881508847|
| 53.49401588571072|
+------------------+
only showing top 5 rows

Activity Tail Values:
+--------------------+
|            activity|
+--------------------+
|7.462960084532733E11|
|7.462960084532733E11|
|7.462960084532733E11|
|7.332167392579166E11|
|7.332167392579166E11|
+--------------------+
only showing top 5 rows

Activity Count:
15184500
Activity Value Counts:
+------------------+-----+
|          activity|count|
+------------------+-----+
|210.16141074308769|    1|
| 219.1031731191849|    1|
| 208.9688036307177|    1|
|429.30412158170657|    1|
| 26.39082389959608|    1|
|285.60865348012084|    1|
| 38.94160095112392|    1|
| 4704.396085228726|    1|
|   266.62472200094|    1|
| 640.8810642718123|    1|
| 3.799954812447411|    1|
|21.817637891104336|    1|
| 82.90283812687248|    1

### <span style="color: white; font-weight: bold; text-decoration: underline;">source_name</span>
The name of the data source.

Data Type: `string`

**Status: <span style="color: green;">READY</span>**

Problems:
1. has null or missing values

In [55]:
# Source Name Exploration
column_title = "Source Name"
column_name = "source_name"

# Data Type
print(f"{column_title} Data Type")
data_type = df.schema[column_name].dataType
print(data_type)

# Head
print(f"\n{column_title} Head Values:")
df.select(column_name).show(5)

# Tail
print(f"{column_title} Tail Values:")
df.select(column_name).orderBy(column_name, ascending=False).show(5)

# Min and Max
if not isinstance(data_type, StringType):
    print(f"{column_title} Max Value:")
    df.agg({column_name : "max"}).show()

    print(f"{column_title} Min Value:")
    df.agg({column_name : "min"}).show()

# Distinct
if isinstance(data_type, StringType):
    print(f"{column_title} Distinct Values:")
    df.select(column_name).distinct().show()

# Total Count
print(f"{column_title} Count:")
print(df.select(column_name).count())

# Value Count
print(f"\n{column_title} Value Counts:")
df.groupBy(column_name).count().show()

# Null Count
print(f"{column_title} Null Count:")
print(df.filter(col(column_name).isNull()).count())

Source Name Data Type
StringType()
Source Name Head Values:
+-----------+
|source_name|
+-----------+
|   Ala-Buka|
|   Ala-Buka|
|   Ala-Buka|
|   Ala-Buka|
|       Alai|
+-----------+
only showing top 5 rows

Source Name Tail Values:
+-----------+
|source_name|
+-----------+
|       웅상|
|       웅상|
|       웅상|
|       웅상|
|       웅상|
+-----------+
only showing top 5 rows

Source Name Distinct Values:
+--------------------+
|         source_name|
+--------------------+
|            Vilniaus|
|            Coapilla|
|Santa Ana Ateixtl...|
|        Villahermosa|
|            Xicotlán|
|           Bar Kunar|
|           Bolyarovo|
|        Borrazópolis|
|              Pongaí|
|    Agapovskiy rayon|
|  Kardymovskiy rayon|
|               Topki|
|         Tejutepeque|
|             Na Thom|
|Le Haut-Saint-Fra...|
|     Le Rocher-Percé|
|            Donggang|
|           Kisangani|
|            Fredonia|
|            Girardot|
+--------------------+
only showing top 20 rows

Source Name Coun

### <span style="color: white; font-weight: bold; text-decoration: underline;">source_type</span>
The industry or service type of source.

Data Type: `string`

**Status: <span style="color: green;">READY</span>**

Problems:
1. has null or missing values

In [56]:
# Source Type Exploration
column_title = "Source Type"
column_name = "source_type"

# Data Type
print(f"{column_title} Data Type")
data_type = df.schema[column_name].dataType
print(data_type)

# Head
print(f"\n{column_title} Head Values:")
df.select(column_name).show(5)

# Tail
print(f"{column_title} Tail Values:")
df.select(column_name).orderBy(column_name, ascending=False).show(5)

# Min and Max
if not isinstance(data_type, StringType):
    print(f"{column_title} Max Value:")
    df.agg({column_name : "max"}).show()

    print(f"{column_title} Min Value:")
    df.agg({column_name : "min"}).show()

# Distinct
if isinstance(data_type, StringType):
    print(f"{column_title} Distinct Values:")
    df.select(column_name).distinct().show()

# Total Count
print(f"{column_title} Count:")
print(df.select(column_name).count())

# Value Count
print(f"\n{column_title} Value Counts:")
df.groupBy(column_name).count().show()

# Null Count
print(f"{column_title} Null Count:")
print(df.filter(col(column_name).isNull()).count())

Source Type Data Type
StringType()
Source Type Head Values:
+-----------+
|source_type|
+-----------+
|       NULL|
|       NULL|
|       NULL|
|       NULL|
|       NULL|
+-----------+
only showing top 5 rows

Source Type Tail Values:
+---------------+
|    source_type|
+---------------+
|zinc_production|
|zinc_production|
|zinc_production|
|zinc_production|
|zinc_production|
+---------------+
only showing top 5 rows

Source Type Distinct Values:
+--------------------+
|         source_type|
+--------------------+
|      DRI-EAF,BF/BOF|
|manufacturer | Li...|
|Meat Processing| ...|
|Certification - E...|
|coal, other_fossi...|
|Certification - E...|
|       biomass, coal|
|Imported Product|...|
|Meat Processing| ...|
|Identification - ...|
|manufacturer | Li...|
|manufacturer | Li...|
|Certification - E...|
|Identification - ...|
|Certification - E...|
|Meat Processing| ...|
|Meat Processing| ...|
|enteric_fermentat...|
|Meat Processing| ...|
|Meat Processing| ...|
+------------------

### <span style="color: white; font-weight: bold; text-decoration: underline;">geometry_ref</span>
Geometry ref of the area being observed

Data Type: `string`

**Status: <span style="color: green;">READY</span>**

Problems: None

In [57]:
# Geometry Reference Exploration
column_title = "Geometry Reference"
column_name = "geometry_ref"

# Data Type
print(f"{column_title} Data Type")
data_type = df.schema[column_name].dataType
print(data_type)

# Head
print(f"\n{column_title} Head Values:")
df.select(column_name).show(5)

# Tail
print(f"{column_title} Tail Values:")
df.select(column_name).orderBy(column_name, ascending=False).show(5)

# Min and Max
if not isinstance(data_type, StringType):
    print(f"{column_title} Max Value:")
    df.agg({column_name : "max"}).show()

    print(f"{column_title} Min Value:")
    df.agg({column_name : "min"}).show()

# Distinct
if isinstance(data_type, StringType):
    print(f"{column_title} Distinct Values:")
    df.select(column_name).distinct().show()

# Total Count
print(f"{column_title} Count:")
print(df.select(column_name).count())

# Value Count
print(f"\n{column_title} Value Counts:")
df.groupBy(column_name).count().show()

# Null Count
print(f"{column_title} Null Count:")
print(df.filter(col(column_name).isNull()).count())

Geometry Reference Data Type
StringType()
Geometry Reference Head Values:
+--------------+
|  geometry_ref|
+--------------+
|gadm_KGZ.4.2_1|
|gadm_KGZ.4.2_1|
|gadm_KGZ.4.2_1|
|gadm_KGZ.4.2_1|
|gadm_KGZ.7.1_1|
+--------------+
only showing top 5 rows

Geometry Reference Tail Values:
+------------+
|geometry_ref|
+------------+
|   trace_923|
|   trace_923|
|   trace_923|
|   trace_923|
|   trace_923|
+------------+
only showing top 5 rows

Geometry Reference Distinct Values:
+-----------------+
|     geometry_ref|
+-----------------+
|     gadm_KGZ.9_1|
|gadm_MEX.20.393_2|
|gadm_MEX.20.413_2|
| gadm_MEX.29.59_2|
|  gadm_AFG.22.2_1|
|     gadm_AND.4_1|
| gadm_ARG.14.14_1|
|   trace_37161875|
|  gadm_BOL.7.16_2|
|  gadm_BRA.4.13_2|
| gadm_ROU.11.71_1|
|   gadm_SAU.3.4_1|
|   gadm_TLS.9.3_1|
|   trace_37178287|
|  gadm_COD.11.5_1|
|  gadm_COD.17.6_1|
|   trace_37188277|
|   gadm_ECU.2.3_1|
|   trace_37234621|
|   trace_37185790|
+-----------------+
only showing top 20 rows

Geometry Refer

### <span style="color: white; font-weight: bold; text-decoration: underline;">Metric Units</span>
Units of measurement used for each metric

Data Type: `string`

**Status: <span style="color: green;">READY</span>**

Problems:
1. The units are the same across all datasets and this overhead can be minimized by adding it to the field's label.

In [9]:
# Metric Units Exploration
metric_fields = {
    "emissions_factor_units": "Emissions Factor Units",
    "capacity_units": "Capacity Units",
    "activity_units": "Activity Units"
}

for each in metric_fields:
    column_title = metric_fields[each]
    column_name = each

    print(f"""{column_name}
================================================================
""")

    print(f"{column_title} Data Type")
    data_type = df.schema[column_name].dataType
    print(data_type)

    print(f"{column_title} Head Values:")
    df.select(column_name).show(5)

    print(f"{column_title} Tail Values:")
    df.select(column_name).orderBy(column_name, ascending=False).show(5)

    if isinstance(data_type, StringType):
        print(f"{column_title} Distinct Values:")
        df.select(column_name).distinct().show()

    print(f"{column_title} Count:")
    print(df.count())

    print(f"{column_title} Value Counts:")
    df.groupBy(column_name).count().show()

    print(f"{column_title} Null Count:")
    print(df.filter(col(column_name).isNull()).count())

emissions_factor_units

Emissions Factor Units Data Type
StringType()
Emissions Factor Units Head Values:
+----------------------+
|emissions_factor_units|
+----------------------+
|  t of CH4 per area...|
|  t of CH4 per area...|
|  t of CH4 per area...|
|  t of CH4 per area...|
|  t of CH4 per area...|
+----------------------+
only showing top 5 rows

Emissions Factor Units Tail Values:
+----------------------+
|emissions_factor_units|
+----------------------+
|  t of CH4 per tonn...|
|  t of CH4 per tonn...|
|  t of CH4 per tonn...|
|  t of CH4 per tonn...|
|  t of CH4 per tonn...|
+----------------------+
only showing top 5 rows

Emissions Factor Units Distinct Values:
+----------------------+
|emissions_factor_units|
+----------------------+
|  t of CH4 per t of...|
|  t of CH4 per tonn...|
|  t of CH4 per t of...|
|  t of CH4 per t of...|
|  t of CH4 per area...|
|  t of CH4 per t of...|
|  t of CH4 per anim...|
|  t of CH4 per t of...|
|  t of CH4 per t of...|
|  t of CH4 per t 

### <span style="color: white; font-weight: bold; text-decoration: underline;">lat and lon</span>
Coordinates of the source

Data Type: `double`

**Status: <span style="color: read;">NOT READY</span>**

Problems:
1. Longitude fields have missing values.
2. Latitude fields have missing values.

In [12]:
# Latitude and Longitude Exploration
metric_fields = {
    "lat": "Latitude",
    "lon": "Longitude"
}

for each in metric_fields:
    column_title = metric_fields[each]
    column_name = each

    print(f"""{column_name}
================================================================
""")

    print(f"{column_title} Data Type")
    data_type = df.schema[column_name].dataType
    print(data_type)

    print(f"{column_title} Head Values:")
    df.select(column_name).show(5)

    print(f"{column_title} Tail Values:")
    df.select(column_name).orderBy(column_name, ascending=False).show(5)

    if isinstance(data_type, StringType):
        print(f"{column_title} Distinct Values:")
        df.select(column_name).distinct().show()

    print(f"{column_title} Count:")
    print(df.count())

    print(f"{column_title} Value Counts:")
    df.groupBy(column_name).count().show()

    print(f"{column_title} Null Count:")
    print(df.filter(col(column_name).isNull()).count())

lat

Latitude Data Type
DoubleType()
Latitude Head Values:
+-----------------+
|              lat|
+-----------------+
|41.41383005758116|
|41.41383005758116|
|41.41383005758116|
|41.41383005758116|
|39.85186444062204|
+-----------------+
only showing top 5 rows

Latitude Tail Values:
+----------------+
|             lat|
+----------------+
|80.7068335565229|
|80.7068335565229|
|80.7068335565229|
|80.7068335565229|
|80.7068335565229|
+----------------+
only showing top 5 rows

Latitude Count:
15184500
Latitude Value Counts:
+-------------------+-----+
|                lat|count|
+-------------------+-----+
| 42.716402701454015|  204|
| 17.396693570568917|  204|
|  19.55551914932812|  204|
|  18.22474547105624|  192|
| 17.213799048555995|  204|
|  35.76691695635098|  204|
|  41.83304253212797|  204|
|  42.36884433678797|  204|
|  25.43935849616272|  192|
|-13.519109790014776|  192|
| -22.44773567020908|  204|
|  44.39115468908935|  204|
|  52.65748874386815|  204|
|  40.16504565450128| 

### <span style="color: white; font-weight: bold; text-decoration: underline;">Other Metadata</span>
Metadata of the each record

Data Type: `string`

**Status: <span style="color: read;">NOT READY</span>**

Problems:
1. Metadata are inconsistent
2. Metadata have massive missing values that cannot be imputed

These inconsistencies have irregular patterns across the datasets.

In [15]:
# Other Fields Value Exploration
metric_fields = {f"other{i + 1}": f"Other{i + 1}" for i in range(6)}

for column_name, column_title in metric_fields.items():
    print(f"""{column_name}
================================================================
""")

    print(f"{column_title} Data Type")
    data_type = df.schema[column_name].dataType
    print(data_type)

    print(f"{column_title} Head Values:")
    df.select(column_name).show(5)

    print(f"{column_title} Tail Values:")
    df.select(column_name).orderBy(column_name, ascending=False).show(5)

    if isinstance(data_type, StringType):
        print(f"{column_title} Distinct Values:")
        df.select(column_name).distinct().show()

    print(f"{column_title} Count:")
    print(df.count())

    print(f"{column_title} Value Counts:")
    df.groupBy(column_name).count().show()

    print(f"{column_title} Null Count:")
    print(df.filter(col(column_name).isNull()).count())

other1

Other1 Data Type
StringType()
Other1 Head Values:
+-------+
| other1|
+-------+
|KGZ.4_1|
|KGZ.4_1|
|KGZ.4_1|
|KGZ.4_1|
|KGZ.7_1|
+-------+
only showing top 5 rows

Other1 Tail Values:
+---------------+
|         other1|
+---------------+
|zinc_production|
|zinc_production|
|zinc_production|
|zinc_production|
|zinc_production|
+---------------+
only showing top 5 rows

Other1 Distinct Values:
+------------------+
|            other1|
+------------------+
|          AFG.28_1|
|          RUS.26_1|
|          TUR.56_1|
|          DOM.24_1|
|          DZA.45_1|
|           ESP.4_1|
|           ZMB.9_1|
|        2.91075952|
|        8.68623224|
|2.3626830135665684|
|0.6074737700000001|
|0.5969723699999999|
|0.5897305899999999|
|0.6107070000000001|
|0.6139374999999999|
|        0.59750472|
|        0.54703157|
|           EGY.3_1|
|          THA.13_1|
|           BTN.1_1|
+------------------+
only showing top 20 rows

Other1 Count:
15184500
Other1 Value Counts:
+------------------+--

### <span style="color: white; font-weight: bold; text-decoration: underline;">confidence metrics</span>
Level of data accuracy measured in confidence

Data Type: `string`

**Status: <span style="color: read;">NOT READY</span>**

Problems:
1. The values are nearly uniform for each dataset. These values can be added as metadata in the documentation to reduce overhead due to redundancy

In [16]:
# Data Confidence Exploration
metric_fields = {
    "conf_source_type": "Source Type Confidence Level",
    "conf_capacity": "Capacity Confidence Level",
    "conf_capacity_factor": "Capacity Factor Confidence Level",
    "conf_activity": "Activity Confidence Level",
    "conf_emissions_factor": "Emissions Factor Confidence Level",
    "conf_emissions_quantity" : "Emissions Quantity Confidence Level"
}

for each in metric_fields:
    column_title = metric_fields[each]
    column_name = each

    print(f"""{column_name}
================================================================
""")

    print(f"{column_title} Data Type")
    data_type = df.schema[column_name].dataType
    print(data_type)

    print(f"{column_title} Head Values:")
    df.select(column_name).show(5)

    print(f"{column_title} Tail Values:")
    df.select(column_name).orderBy(column_name, ascending=False).show(5)

    if isinstance(data_type, StringType):
        print(f"{column_title} Distinct Values:")
        df.select(column_name).distinct().show()

    print(f"{column_title} Count:")
    print(df.count())

    print(f"{column_title} Value Counts:")
    df.groupBy(column_name).count().show()

    print(f"{column_title} Null Count:")
    print(df.filter(col(column_name).isNull()).count())

conf_source_type

Source Type Confidence Level Data Type
StringType()
Source Type Confidence Level Head Values:
+----------------+
|conf_source_type|
+----------------+
|        very low|
|        very low|
|        very low|
|        very low|
|        very low|
+----------------+
only showing top 5 rows

Source Type Confidence Level Tail Values:
+----------------+
|conf_source_type|
+----------------+
|        very low|
|        very low|
|        very low|
|        very low|
|        very low|
+----------------+
only showing top 5 rows

Source Type Confidence Level Distinct Values:
+----------------+
|conf_source_type|
+----------------+
|             low|
|        very low|
|            high|
|          medium|
|       very high|
+----------------+

Source Type Confidence Level Count:
15184500
Source Type Confidence Level Value Counts:
+----------------+--------+
|conf_source_type|   count|
+----------------+--------+
|             low|  122064|
|        very low|13857291|
|       